# ASHRAE Agentic Scraper — Google Colab

Notebook-native entry point for the repository's shared Python pipeline. No GPU/TPU, Streamlit server or tunnel is needed.

**Pipeline:** Search → Stage 1 query filter → existing-local-document check → download/recovery → full-PDF local retrieval → CrewAI Stage 2 → ASHRAE identity + query relevance → accepted library.

Run sections in order. Credentials stay in environment variables; notebook outputs never display them. Local runtime data disappears when Colab resets unless you enable Drive persistence. Keep the notebook and its source modules from the same revision.

## 1. Environment check

In [ ]:
import os, platform, shutil, sys
from pathlib import Path
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('CPUs:', os.cpu_count())
try:
    mem = dict(line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines())
    print('Available RAM: %.2f GiB' % (int(mem['MemAvailable'].split()[0]) / 1024**2))
except (OSError, KeyError):
    print('Available RAM: unavailable on this platform')
disk = shutil.disk_usage('/content' if Path('/content').exists() else Path.cwd())
print('Available local disk: %.2f GiB' % (disk.free / 1024**3))
print('CPU runtime is sufficient; GPU/TPU is not required.')

Python: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
CPUs: 2
Available RAM: 11.43 GiB
Available local disk: 85.96 GiB
CPU runtime is sufficient; GPU/TPU is not required.


## 2. Clone / update / open repository

Default: clone on first run, then `git pull --ff-only` on clean later runs. Dirty worktrees are preserved. Set `REPO_REF` to the branch containing these changes once published.

**Before these changes are published:** upload `Agentic_Scraper_Colab_source.zip` using Colab's Files sidebar and set `SOURCE_ARCHIVE` to `/content/Agentic_Scraper_Colab_source.zip`. The bundle opens in a new repository directory. Existing directories are never overwritten by bundle extraction. Use another `REPO_DIR` if needed.

In [ ]:
import subprocess, zipfile
REPO_URL = 'https://github.com/haqueWasif/Agentic-Scrapper.git'
REPO_REF = 'main'
REPO_DIR = Path('/content/Agentic-Scrapper-workers')
SOURCE_ARCHIVE = '/content/Agentic_Scraper_Colab_source.zip'

if not REPO_DIR.exists():
    if SOURCE_ARCHIVE:
        REPO_DIR.mkdir(parents=True)
        with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
            for member in archive.infolist():
                target = (REPO_DIR / member.filename).resolve()
                if REPO_DIR.resolve() not in target.parents:
                    raise ValueError('Unsafe source archive path')
            archive.extractall(REPO_DIR)
        print('Opened uploaded source bundle.')
    else:
        subprocess.run(['git', 'clone', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
elif (REPO_DIR / '.git').exists():
    dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip()
    branch = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
    if dirty or branch != REPO_REF:
        print('Using existing checkout; local changes or a different branch prevent automatic update.')
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    print('Using existing uploaded source directory.')
if not (REPO_DIR / 'colab' / 'pipeline_runner.py').is_file():
    raise RuntimeError('This checkout lacks the Colab adapter. Use the source bundle or a branch containing the Colab changes.')
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Repository:', REPO_DIR)

Using existing uploaded source directory.
Repository: /content/Agentic-Scrapper-workers


## 3. Install the existing requirements

This uses `requirements.txt`, the project's only dependency list. It includes Streamlit for local users, but notebook execution neither imports it nor starts a server. No browser automation process is needed by this pipeline's direct search/download path. If Colab requests a runtime restart after installation, restart once and rerun the setup cells.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements.txt')], check=True)
print('Requirements installed.')

Requirements installed.


## 4. Secrets and optional LangSmith tracing

In [ ]:
from getpass import getpass
LANGSMITH_TRACING = False
LANGSMITH_PROJECT = 'ASHRAE-Colab'
USE_ZENROWS = True  # Optional existing provider; direct fallback is preserved.

def configure_secret(name, required=False):
    value = os.environ.get(name, '')
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name) or ''
        except Exception:
            value = ''
    if not value and required:
        value = getpass(name + ': ')
    if value:
        os.environ[name] = value
    elif required:
        raise ValueError(name + ' is required')

configure_secret('OPENROUTER_API_KEY', required=True)
configure_secret('ZENROWS_API_KEY', required=USE_ZENROWS)
configure_secret('LANGSMITH_API_KEY', required=LANGSMITH_TRACING)
os.environ['LANGSMITH_TRACING'] = str(LANGSMITH_TRACING).lower()
os.environ['LANGSMITH_PROJECT'] = LANGSMITH_PROJECT
print('Secrets configured. LangSmith tracing:', LANGSMITH_TRACING)

OPENROUTER_API_KEY: ··········
ZENROWS_API_KEY: ··········
Secrets configured. LangSmith tracing: False


## 5. Notebook configuration

The existing project settings provide queue/retry defaults. This adapter injects the settings below without changing `config/settings.yaml`. Defaults are five download workers and two validation workers. Explicit positive-integer overrides are supported (for example, 50 downloads and 10 validations), but no higher count has been validated for remote throughput or your runtime's memory capacity. Counts never increase automatically. Start pacing remains **2–6 seconds**. Changing Colab hardware cannot fix a remote HTTP 500/503 or a source cooldown.

In [ ]:
QUERY = 'ASHRAE'
MAX_PAGES = 50
TARGET_DOCUMENTS = 500
DOWNLOAD_WORKERS = 50
STAGE2_WORKERS = 10
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/ASHRAE_Scraper'
LOCAL_WORK_DIR = '/content/ashrae_work'
EXISTING_LIBRARY_DIRS = []  # Optional: ['/content/drive/MyDrive/My existing ASHRAE library']
PROGRESS_STATE_INTERVAL_SECONDS = 2.0
PROGRESS_STATE_BYTES_MB = 8
PROGRESS_EVENT_INTERVAL_SECONDS = 0.5
DOWNLOAD_CHUNK_SIZE = 256 * 1024
NEW_DOWNLOAD_MIN_SECONDS = 2.0
NEW_DOWNLOAD_MAX_SECONDS = 6.0
CHECKPOINT_PARTIALS_TO_DRIVE = False
PARTIAL_CHECKPOINT_SECONDS = 600
DEBUG_VERBOSE = False
print(f'Workers: {DOWNLOAD_WORKERS} download / {STAGE2_WORKERS} Stage 2')
print(f'Download start pacing: {NEW_DOWNLOAD_MIN_SECONDS:g}–{NEW_DOWNLOAD_MAX_SECONDS:g} seconds')

Workers: 50 download / 10 Stage 2
Download start pacing: 2–6 seconds


## 6. Fast active storage and optional Google Drive

Network → local `/content/ashrae_work/*.part` → completed PDF → coarse persistence step.

Drive receives completed PDFs and state snapshots every minute and at completion/interruption. Partial checkpoints are disabled by default; enabling them copies a bounded prefix at most every ten minutes, plus the final checkpoint. A large partial can be expensive to copy. No Drive operation runs in a network chunk callback.

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
Path(LOCAL_WORK_DIR).mkdir(parents=True, exist_ok=True)
print('Active local storage:', LOCAL_WORK_DIR)
print('Persistent storage:', DRIVE_OUTPUT_DIR if USE_GOOGLE_DRIVE else 'disabled')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Active local storage: /content/ashrae_work
Persistent storage: /content/drive/MyDrive/ASHRAE_Scraper


## 7. Initialize shared modules and persistent resume

The execution call restores missing state, the cached library index, validation reports, and completed PDF references from Drive. Existing local files win, preserving newer progress when you rerun a cell. Completed PDFs are linked for on-demand reuse rather than recopied at every startup. New or changed library files receive metadata indexing; unchanged files reuse cached metadata. Full page extraction happens only during Stage 2.

If partial checkpoints are enabled, real `.part` snapshots are copied onto local disk. **Their actual filesystem size is the HTTP Range offset**, even when `downloads.json` contains older progress. With partial checkpoints disabled, interrupted bytes survive only within the current Colab runtime; completed data and state still persist.

In [ ]:
from app.download_progress import ProgressPolicy
from colab.pipeline_runner import ColabConfig, NotebookProgress, run_colab_pipeline
CONFIG = ColabConfig(
    local_work_dir=LOCAL_WORK_DIR, use_google_drive=USE_GOOGLE_DRIVE,
    drive_output_dir=DRIVE_OUTPUT_DIR, existing_library_dirs=tuple(EXISTING_LIBRARY_DIRS),
    download_workers=DOWNLOAD_WORKERS, stage2_workers=STAGE2_WORKERS,
    new_download_min_seconds=NEW_DOWNLOAD_MIN_SECONDS, new_download_max_seconds=NEW_DOWNLOAD_MAX_SECONDS,
    progress_policy=ProgressPolicy(PROGRESS_STATE_INTERVAL_SECONDS, PROGRESS_STATE_BYTES_MB * 1024**2,
                                   PROGRESS_EVENT_INTERVAL_SECONDS, DOWNLOAD_CHUNK_SIZE),
    checkpoint_partials_to_drive=CHECKPOINT_PARTIALS_TO_DRIVE,
    partial_checkpoint_seconds=PARTIAL_CHECKPOINT_SECONDS, debug_verbose=DEBUG_VERBOSE,
)
print('Configuration ready; restore happens before search in the next cell.')

Configuration ready; restore happens before search in the next cell.


## 8. Run the shared pipeline

One updating output area, no dashboard server. Use the cell Stop button to interrupt; allow bounded in-flight requests to finish and the final checkpoint to complete before starting another run. Technical details go to `LOCAL_WORK_DIR/logs/scraper.log`. No per-page LLM calls or whole-PDF uploads: CrewAI evaluates bounded publisher/query evidence retrieved from all extractable pages.

In [ ]:
result = await run_colab_pipeline(
    query=QUERY, max_pages=MAX_PAGES, target_documents=TARGET_DOCUMENTS,
    config=CONFIG, on_snapshot=NotebookProgress(),
)

[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

...

[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

## 9. Results and output directories

In [ ]:
labels = {
    'pages_processed': 'Pages processed', 'candidates_discovered': 'Candidates discovered',
    'documents_approved': 'Stage 1 approved', 'stage1_rejected': 'Stage 1 rejected',
    'local_pdfs_reused': 'Local PDFs reused', 'new_pdfs_downloaded': 'New PDFs downloaded',
    'download_failures': 'Download failures', 'recovery_remaining': 'Recovery remaining',
    'ashrae_approved': 'ASHRAE identity approved', 'query_relevant': 'Query relevant',
    'stage2_approved': 'Accepted this query', 'pending_validation': 'Pending validation', 'rejected': 'Rejected',
}
for key, label in labels.items():
    print(f'{label}: {result.get(key, 0)}')
for folder in ('ASHRAE_Files', 'approved', 'rejected', 'validation_reports', 'logs'):
    print(folder + ':', Path(LOCAL_WORK_DIR) / folder)
print('Existing library references:', EXISTING_LIBRARY_DIRS)
print('Persistent output:', result['persistent_directory'])

## 10. Optional export or individual PDF download

Export reports and selected metadata only. PDFs are never bulk-zipped automatically. For a very large library, use Drive instead of browser downloads. Reused legacy PDFs retain their original filenames and locations; reports record their paths.

In [ ]:
EXPORT_REPORTS = False
SELECTED_PDF = ''  # Full path of one PDF to download, or leave empty.
if EXPORT_REPORTS:
    export_path = Path(LOCAL_WORK_DIR) / 'validation_metadata.zip'
    with zipfile.ZipFile(export_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in (Path(LOCAL_WORK_DIR) / 'validation_reports').glob('*.json'):
            archive.write(path, path.relative_to(LOCAL_WORK_DIR))
        for name in ('downloads.json', 'local_library_index.json', 'run_summary.json'):
            path = Path(LOCAL_WORK_DIR) / name
            if path.exists():
                archive.write(path, name)
    from google.colab import files
    files.download(str(export_path))
if SELECTED_PDF:
    selected = Path(SELECTED_PDF)
    if selected.suffix.lower() != '.pdf' or not selected.is_file():
        raise ValueError('Select an existing PDF')
    from google.colab import files
    files.download(str(selected))

## 11. Optional performance benchmark

Run while the pipeline is idle. The synthetic test exercises the real binary writer, ledger, and progress policy with 100 MiB at 256/512/1024 KiB chunks. It measures **local overhead only**, not remote bandwidth. Keep 256 KiB by default: synthetic disk speed alone does not justify changing network chunk size or worker counts.

The real run totals below separately report bytes received (excluding pre-existing resume bytes), time inside binary streaming including local callback overhead, durable writes/write time, progress events, retries/cooldowns, start pacing, and mirror HTTP resolution time. For five concurrent workers, transfer seconds are the sum of worker times, not wall-clock throughput. Compare the same authorized workload on Windows and Colab. High ledger time suggests persistence overhead; high retry/pacing time suggests waiting; low binary throughput can reflect remote delivery or bandwidth and cannot by itself distinguish them. No claim that Colab is faster is made.

In [ ]:
RUN_LOCAL_BENCHMARK = False
if RUN_LOCAL_BENCHMARK:
    from colab.download_benchmark import benchmark_local_io
    for measurement in benchmark_local_io(directory=LOCAL_WORK_DIR):
        print(measurement)
if 'result' in globals():
    print('Real pipeline totals:', result['benchmark_totals'])